# Inner products and SNRs

This example uses SEF's lower-level utilities on their own, without `StableEMRIFisher`. It generates a waveform through the LISA response with FEW and `fastlisaresponse`, builds the noise PSD of each TDI channel with `generate_psd`, and computes the signal-to-noise ratio $\rho = \sqrt{(h|h)}$ with `inner_product`.

The noise-weighted inner product of two signals $a$ and $b$ is

$$(a|b) = 4\,\mathrm{Re} \sum_{\rm channels} \int_0^\infty \frac{\tilde a^*(f)\, \tilde b(f)}{S_n(f)}\, df,$$

where $\tilde a(f)$ is the Fourier transform of $a$ and $S_n(f)$ is the noise PSD of each channel.

**Requirements:** `lisaanalysistools` and `fastlisaresponse`, for the LISA response and noise model (see the installation guide).

*Validated with LISAAnalysisTools-cuda12x v1.2.0 and fastlisaresponse v1.1.17.*

In [1]:
import time

import numpy as np

import few
from few.waveform import GenerateEMRIWaveform, FastKerrEccentricEquatorialFlux
from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens

from stableemrifisher.utils import generate_psd, inner_product, fourier_coefficients, inner_product_fd

use_gpu = False  # set to True to run on an NVIDIA GPU

if not use_gpu:
    # restrict FEW to its CPU backend
    cfg_set = few.get_config_setter(reset=True)
    cfg_set.enable_backends("cpu")

if use_gpu:
    import cupy as xp  # waveforms are CuPy arrays on the GPU
else:
    xp = np

## Source parameters

A prograde, equatorial EMRI around a spinning black hole, observed for 0.1 years.

In [2]:
T = 0.1  # observation time, years
dt = 10.0  # sampling interval, seconds

wave_params = dict(
    m1=1e6,  # primary mass, solar masses
    m2=10.0,  # secondary mass, solar masses
    a=0.9,  # primary spin
    p0=10.0,  # initial semi-latus rectum
    e0=0.1,  # initial eccentricity
    xI0=1.0,  # cosine of the initial inclination (1 for this equatorial model)
    dist=1.0,  # luminosity distance, Gpc
    qS=np.pi / 6,  # sky location, polar angle
    phiS=np.pi / 8,  # sky location, azimuthal angle
    qK=np.pi / 3,  # spin orientation, polar angle
    phiK=np.pi / 4,  # spin orientation, azimuthal angle
    Phi_phi0=0.0,  # initial azimuthal phase
    Phi_theta0=0.0,  # initial polar phase
    Phi_r0=0.0,  # initial radial phase
)

## Waveform model with the LISA response

FEW's Kerr eccentric equatorial flux model, wrapped in the LISA TDI response (first generation, equal-arm-length orbits), returning the A, E and T channels.

In [3]:
waveform_model = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux,
    sum_kwargs=dict(pad_output=True),  # zero-pad to the full observation time if the inspiral plunges
    inspiral_kwargs=dict(err=1e-11),  # trajectory integrator tolerance
    return_list=False,  # return h+ - i hx as one complex array, as the response expects
)

waveform_response = ResponseWrapper(
    waveform_gen=waveform_model,
    Tobs=T,
    dt=dt,
    index_lambda=8,  # position of phiS in the waveform arguments
    index_beta=7,  # position of qS in the waveform arguments
    t0=10000.0,  # seconds at each end of the signal where the response is unreliable
    remove_garbage="zero",  # zero the signal within t0 of each end
    flip_hx=True,  # FEW returns h+ - i hx
    is_ecliptic_latitude=False,  # qS is a polar angle, not a latitude
    orbits=EqualArmlengthOrbits(),
    order=20,  # order of the Lagrange interpolation of the time delays
    tdi="1st generation",
    tdi_chan="AET",
)

wave = xp.asarray(waveform_response(*wave_params.values(), T=T, dt=dt))
print("waveform shape (channels, samples):", wave.shape)

waveform shape (channels, samples): (3, 315581)


## PSDs and SNR

The noise model is `get_sensitivity` from `lisatools`, with one `sens_fn` per TDI channel, in the same order as the channels of the response. `generate_psd` evaluates it on the frequency grid of the waveform's Fourier transform, giving one PSD array per channel. The SNR is the square root of the inner product of the waveform with itself, summed over the channels.

In [4]:
channels = [A1TDISens, E1TDISens, T1TDISens]

PSD = generate_psd(
    wave,
    dt=dt,
    noise_PSD=get_sensitivity,
    channels=channels,
    noise_kwargs=[{"sens_fn": channel} for channel in channels],
    use_gpu=use_gpu,
)

snr = np.sqrt(inner_product(wave, wave, PSD, dt, use_gpu=use_gpu))
print(f"SNR = {snr:.2f}")

# contribution of each channel; the squares add up to SNR^2
for name, h, S in zip("AET", wave, PSD):
    print(f"  {name} channel: SNR = {np.sqrt(inner_product(h, h, S, dt, use_gpu=use_gpu)):.2f}")

SNR = 9.03
  A channel: SNR = 8.37
  E channel: SNR = 3.37
  T channel: SNR = 0.02


## Extension: computing the Fourier transform once

The inner product is computed in two stages:

1. **Transform:** the Fourier coefficients $\tilde a_c(f) = \Delta t\, \mathrm{rfft}(a_c)$ of each input (`fourier_coefficients`). This is the expensive stage, one FFT per channel.
2. **Weighted product:** $4\,\Delta f\, \mathrm{Re} \sum_c \sum_f \tilde a_c^*(f)\, \tilde b_c(f) / S_c(f)$ (`inner_product_fd`), an element-wise product and a sum.

The coefficients of a time series depend only on that time series, so they can be computed once and reused. A naive implementation transforms both inputs of every inner product: for $(h|h)$ it computes the same FFT twice. The staged implementation computes it once. In a Fisher matrix with $n$ parameters, each derivative enters $n + 1$ of the computed elements $\Gamma_{ij}$ ($i \le j$), so StableEMRIFisher transforms each derivative once and reuses its coefficients, which saves a factor of $n + 1$ in FFTs.

Below, both implementations compute $(h|h)$ for the waveform above, and their cost is averaged over 100 runs.

In [6]:
def naive_inner_product(a, b, PSD, dt):
    """(a|b), Fourier transforming both inputs (twice the same FFT if b is a)."""
    a_f = dt * xp.fft.rfft(a, axis=-1)[:, 1:]
    b_f = dt * xp.fft.rfft(b, axis=-1)[:, 1:]
    df = 1 / (a.shape[-1] * dt)
    return float(4 * df * ((a_f.conj() * b_f).real / xp.asarray(PSD)).sum())


def staged_inner_product(a, PSD, dt):
    """(a|a) in two stages: one Fourier transform, then the weighted product."""
    a_f = fourier_coefficients(a, dt, use_gpu=use_gpu)
    return float(inner_product_fd(a_f, a_f, PSD, use_gpu=use_gpu))


def average_time(func, n_runs=100):
    """Mean and standard deviation of the run time of func(), in seconds."""
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        func()
        if use_gpu:
            xp.cuda.Device().synchronize()  # wait for the GPU to finish
        times.append(time.perf_counter() - start)
    return np.mean(times), np.std(times)


naive = naive_inner_product(wave, wave, PSD, dt)
staged = staged_inner_product(wave, PSD, dt)
print(f"(h|h) naive:  {naive:.10e}")
print(f"(h|h) staged: {staged:.10e}   (relative difference {abs(staged - naive) / naive:.1e})")

# the coefficients of h, computed once and reused by the product-only timing
h_f = fourier_coefficients(wave, dt, use_gpu=use_gpu)

timings = {
    "naive (2 FFTs)": average_time(lambda: naive_inner_product(wave, wave, PSD, dt)),
    "staged (1 FFT)": average_time(lambda: staged_inner_product(wave, PSD, dt)),
    "product only (coefficients reused)": average_time(lambda: inner_product_fd(h_f, h_f, PSD, use_gpu=use_gpu)),
}

print()
for name, (mean, std) in timings.items():
    print(f"{name:36s} {1e3 * mean:7.2f} +/- {1e3 * std:5.2f} ms")

(h|h) naive:  8.1470551071e+01
(h|h) staged: 8.1470551071e+01   (relative difference 1.7e-16)

naive (2 FFTs)                         63.85 +/-  4.60 ms
staged (1 FFT)                         35.44 +/-  1.31 ms
product only (coefficients reused)      2.40 +/-  0.23 ms


## What this shows

`generate_psd` and `inner_product` are the building blocks StableEMRIFisher uses for its SNR and Fisher matrices: each Fisher element is the inner product of two waveform derivatives, $\Gamma_{ij} = (\partial_i h | \partial_j h)$. The T channel contributes almost nothing at these frequencies, which is why the other examples use only A and E.

Most of the cost of an inner product is the Fourier transform. Computing it once per time series halves the cost of $(h|h)$, and once the coefficients are known, further inner products with them cost only the cheap weighted product.